# INF Paper — Reproducibility Supplement

This notebook reproduces the following publication outputs (manuscript and supplementary numbering):
- **Figure 11**: Top-5 metapath features ranked by permutation importance for TPR (recall), evaluated on the Soldatos external test set, first degree decile (0–5 known AEs per target), INF configuration 12
- **Figure S4** (`figures/figS4_holdout_vs_external.png`): Holdout vs. external test performance comparing Hits@10 and TPR across INF ensemble, DWPC, and raw L3 baselines
- **Figure 7** (`figures/fig7_cv_d1_tpr_vertical/horizontal.png`): CV TPR for the 1st decile (0–5 AEs per target) comparing Original, DWPC, and Best INF configuration
- **Figure 8** (`figures/fig8_holdout_d1_vertical/horizontal.png`): Holdout test set 1st decile — filtered Hits@10 and TPR for INF ensemble, DWPC, and Original
- **Figure 9** (`figures/fig9_soldatos_vertical/horizontal.png`): External validation (Soldatos et al.) — filtered Hits@10 and TPR for INF ensemble, DWPC, and Original

## Metric note
Hits@k uses the **filtered** evaluation protocol standard in KGE literature: for each test pair (h, t), the rank of t is computed after removing other known positive tails ranked above it. The score is the fraction of test pairs with filtered rank ≤ k. Recall (TPR) uses a cluster-bootstrap over per-gene TP/FN records.

## Requirements
- Python 3.10, dependencies installed via `.venv/bin/pip install -r requirements.txt` (or equivalent)
- All pre-computed results are cached in `data/pickles/` — no Neo4j instance required
- Soldatos (2018) supplementary files in `data/soldatos_2018/`

## Execution
```
.venv/bin/jupyter nbconvert --to notebook --execute --inplace supplement.ipynb
```
Expected runtime: ~25 minutes (dominated by permutation importance in the Figure 11 cell and ensemble re-training for Figures 7–9).

In [ ]:
# imports
import pickle, pandas as pd, numpy as np
from matplotlib import pyplot as plt
from matplotlib.transforms import blended_transform_factory
import seaborn as sns
from tqdm import tqdm
import re
import copy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, accuracy_score, f1_score, make_scorer
from sklearn.model_selection import ParameterGrid
from scipy import stats
from scipy.stats import t
from scipy.interpolate import make_interp_spline
from typing import List
from collections import defaultdict
from metapaths.starterpack import load_pickle, save_pickle, cypher_triple_to_list, metapath_featset_gen
from metapaths.inf import INFToolbox, query_templates_234, Graph
from typing import Dict, Any, List, Tuple, Optional
import config

In [ ]:

rel_types_cypher = ["(:Gene)-[:Gene_Gene]->(:Gene)",
"(:Gene)-[:Gene_Disease]->(:Disease)",
"(:Gene)-[:Gene_MolecularFunction]->(:MolecularFunction)",
"(:Gene)-[:Gene_BiologicalProcess]->(:BiologicalProcess)",
"(:Gene)-[:Gene_CellularComponent]->(:CellularComponent)",
"(:Gene)-[:Gene_Pathway]->(:Pathway)",
"(:Gene)-[:Gene_Anatomy]->(:Anatomy)",
"(:Gene)-[:Gene_Compound]->(:Compound)",
"(:Disease)-[:Disease_Disease]->(:Disease)",
"(:Compound)-[:Compound_Disease]->(:Disease)",
"(:Compound)-[:Compound_SideEffect]->(:SideEffect)",
"(:CellularComponent)-[:CellularComponent_CellularComponent]->(:CellularComponent)",
"(:BiologicalProcess)-[:BiologicalProcess_BiologicalProcess]->(:BiologicalProcess)",
"(:MolecularFunction)-[:MolecularFunction_MolecularFunction]->(:MolecularFunction)",
"(:Disease)-[:Disease_Symptom]->(:Symptom)",
"(:Compound)-[:Compound_Compound]->(:Compound)",
"(:ADR)-[:ADR_Disease]->(:Disease)",
"(:ADR)-[:ADR_Symptom]->(:Symptom)",
"(:Gene)-[:Gene_ADR]->(:ADR)",
"(:ADR)-[:ADR_SideEffect]->(:SideEffect)"]

len(rel_types_cypher)

In [ ]:
pickles_dir = str(config.PICKLES_DIR)

In [ ]:
# load correct db — Neo4j connection is optional; all queries are pickle-guarded
try:
    toolbox = INFToolbox(Graph(config.NEO4J_URI_READ), query_templates_234)
    _toolbox_ok = True
except Exception as _neo4j_err:
    print(f"Neo4j unavailable ({type(_neo4j_err).__name__}): {_neo4j_err}")
    print("All pre-computed pickles will be used; toolbox set to None.")
    toolbox = None
    _toolbox_ok = False

param_combos_load = load_pickle(f'{pickles_dir}/param_combos.pkl')
if _toolbox_ok:
    toolbox.add_param_combos(param_combos_load)

In [ ]:
all_genes_aes = load_pickle(f'{pickles_dir}/all_genes_with_aes.pkl')

In [ ]:
def ci_margin_of_error(data: pd.Series, confidence: float = 0.95) -> float:
    """Helper function to calculate the confidence interval margin of error."""
    # CI requires at least 2 data points
    if len(data) < 2:
        return 0.0
    
    # Calculate sample standard deviation (ddof=1) and count
    std_dev = np.std(data, ddof=1)
    n = len(data)
    
    # Get the t-score
    t_score = t.ppf((1 + confidence) / 2, df=n - 1)
    
    # Calculate the margin of error
    margin = t_score * (std_dev / np.sqrt(n))
    return margin

In [ ]:
def analyse_parameter_slices_quantile(fold_results: dict, num_slices: int = 10, balance_slices: bool = False, parameter_name: str = 'Gene_ADR_deg', confidence_level: float = 0.95):
    """
    Amended: Analyzes performance by slicing positives into quantiles.
    """
    slice_data = []
    RANDOM_SEED = 69
    for fold_id, fold_data in fold_results.items():
        fold_df = pd.DataFrame({
            'y_val': fold_data['y_val'],
            'y_pred': fold_data['y_pred'],
            'param_values': fold_data.get(parameter_name, fold_data.get('parameter_values'))
        })
        pos_df = fold_df[fold_df['y_val'] == 1].copy()
        if pos_df.empty: continue

        try:
            pos_df['slice_num'] = pd.qcut(pos_df['param_values'], q=num_slices, labels=False, duplicates='drop')
        except ValueError: continue

        for slice_id in sorted(pos_df['slice_num'].unique()):
            slice_pos_df = pos_df[pos_df['slice_num'] == slice_id]
            min_param, max_param = slice_pos_df['param_values'].min(), slice_pos_df['param_values'].max()
            n_pos = len(slice_pos_df)
            slice_neg_conformant = fold_df[(fold_df['y_val'] == 0) & (fold_df['param_values'].between(min_param, max_param))]
            final_slice_negs = slice_neg_conformant.sample(n=n_pos, random_state=RANDOM_SEED) if balance_slices and len(slice_neg_conformant) > n_pos else slice_neg_conformant
            final_slice_df = pd.concat([slice_pos_df, final_slice_negs])
            y_true, y_pred = final_slice_df['y_val'], final_slice_df['y_pred']
            
            slice_data.append({
                'slice_num': slice_id + 1, 'recall': recall_score(y_true, y_pred, zero_division=0),
                'precision': precision_score(y_true, y_pred, zero_division=0), 'f_1': f1_score(y_true, y_pred, zero_division=0),
                'min_param': min_param, 'max_param': max_param, 'true_pos': np.sum((y_pred == 1) & (y_true == 1)),
                'pred_pos': np.sum(y_pred), 'val_neg': len(final_slice_negs), 'pos': n_pos
            })

    if not slice_data: return pd.DataFrame()
    df = pd.DataFrame(slice_data)
    
    cols_to_convert = ['recall', 'precision', 'f_1', 'true_pos', 'pred_pos', 'val_neg', 'pos', 'min_param', 'max_param']
    for col in cols_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=[c for c in cols_to_convert if c in df.columns], inplace=True)
    if df.empty: return pd.DataFrame()

    ci_func = lambda x: ci_margin_of_error(x, confidence=confidence_level)
    result = df.groupby('slice_num').agg(
        recall_mean=('recall', 'mean'), recall_ci=('recall', ci_func),
        precision_mean=('precision', 'mean'), precision_ci=('precision', ci_func),
        f1_mean=('f_1', 'mean'), f1_ci=('f_1', ci_func),
        mean_true_pos=('true_pos', 'mean'), mean_pred_pos=('pred_pos', 'mean'),
        mean_val_neg=('val_neg', 'mean'), mean_pos_per_slice=('pos', 'mean'),
        min_param=('min_param', 'mean'), max_param=('max_param', 'mean')
    ).reset_index()

    result[['min_param', 'max_param']] = result[['min_param', 'max_param']].round(0).astype(int)
    return result.round(3)

In [ ]:
def find_best_configs_quantile(config_results_dict, parameter_name, rank,
    num_slices: int = 10, balance_slices: bool = False, metric: str = 'recall_mean'):
    """
    Amended: Finds the Nth best-performing config for each parameter slice.
    """
    if not isinstance(rank, int) or rank < 1:
        raise ValueError("Rank must be a positive integer (1 or greater).")

    all_configs_data = []
    for config_idx, (_, fold_results, _) in config_results_dict.items():
        slice_df = analyse_parameter_slices_quantile(fold_results, num_slices, balance_slices, parameter_name)
        if slice_df.empty: continue
        
        slice_df.insert(1, 'param_combo', config_idx)
        all_configs_data.append(slice_df)
        
    if not all_configs_data: return pd.DataFrame()
    combined_df = pd.concat(all_configs_data, ignore_index=True)
    if combined_df.empty or combined_df[metric].isna().all(): return pd.DataFrame()
        
    return (
        combined_df.sort_values(by=metric, ascending=False)
        .groupby('slice_num').nth(rank - 1).reset_index()
    )

In [ ]:
from pathlib import Path as _Path
if _Path(config.CV_GS_RESULTS_L3_PKL).exists():
    inf_cv_gs_results_l3 = load_pickle(str(config.CV_GS_RESULTS_L3_PKL))
    print(f'Pre-computed — loaded {config.CV_GS_RESULTS_L3_PKL.name}')
else:
    # Recall
    inf_cv_gs_results_l3 = {}

    for combo_idx, combo in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

        inf_cv_gs_results_l3[combo_idx] = cv_gs_loop(folds_l3_inf_feats_combos[combo_idx], 'Gene_ADR_deg', ['Gene','ADR'])


    save_pickle(inf_cv_gs_results_l3, str(config.CV_GS_RESULTS_L3_PKL))

### Prepare INF data

In [ ]:
inf_test_feats_dir = str(config.INF_FEATS_DIR)

In [ ]:
negs_to_test_pkl = load_pickle(f'{inf_test_feats_dir}/negs_to_test.pkl')

In [ ]:
negs_to_test = copy.deepcopy(negs_to_test_pkl)

In [ ]:
train_test_pkl = load_pickle(f'{inf_test_feats_dir}/train_test.pkl')

In [ ]:
inf_hitk_train_test_feats = {'train_pos': train_test_pkl['pos_train'],
                            'train_neg': train_test_pkl['neg_train'],
                            'negs_to_test': negs_to_test}

In [ ]:
gene_match_adr_query = '''
MATCH (n:Gene)-[rel:Gene_ADR]->() WHERE n.name = $gene RETURN COUNT(rel) AS gene_adr_deg'''

In [ ]:
from pathlib import Path as _Path
if not _Path(config.HITK_PARTITIONS_PKL).exists():
    genes_degs_adr = {}
                    
    for gene in tqdm(all_genes_aes):

        genes_degs_adr[gene] = toolbox.graph.run(gene_match_adr_query, gene=gene).evaluate()

    print('AE per Gene counting (w/ test edges deleted) complete')

    genes_degs_adr_df = pd.Series(genes_degs_adr).reset_index().rename(columns={'index': 'Gene', 0: 'Gene_ADR_deg'})

    genes_degs_adr_df.fillna(0, inplace=True)

    assert not genes_degs_adr_df.isna().any().any(), 'Nans found!'
else:
    print(f'Pre-computed — skipping ({config.HITK_PARTITIONS_PKL.name} exists)')

In [ ]:
from pathlib import Path as _Path
if _Path(config.HITK_PARTITIONS_PKL).exists():
    inf_hitk_train_test_feats = load_pickle(str(config.HITK_PARTITIONS_PKL))
    print(f'Pre-computed — loaded {config.HITK_PARTITIONS_PKL.name}')
else:
    for partition_key in inf_hitk_train_test_feats:

        print(f'Doing {partition_key}...')

        for combo_idx, _ in tqdm(enumerate(toolbox.param_combos), total=len(toolbox.param_combos)):

            if 'Gene' and 'ADR' not in inf_hitk_train_test_feats[partition_key][combo_idx].columns:

                inf_hitk_train_test_feats[partition_key][combo_idx] = toolbox.restore_pair_node_id_cols(inf_hitk_train_test_feats[partition_key][combo_idx],
                                                                                                        'Gene', 0, 'ADR', 1)
        
            if 'Gene_ADR_deg' in inf_hitk_train_test_feats[partition_key][combo_idx].columns:

                inf_hitk_train_test_feats[partition_key][combo_idx] = inf_hitk_train_test_feats[partition_key][combo_idx].drop(columns=['Gene_ADR_deg'])
        
            inf_hitk_train_test_feats[partition_key][combo_idx] = pd.merge(inf_hitk_train_test_feats[partition_key][combo_idx],
                                                                                genes_degs_adr_df, how='left', on='Gene')

In [ ]:
if not (config.HITK_PARTITIONS_PKL).exists():
    save_pickle(inf_hitk_train_test_feats, str(config.HITK_PARTITIONS_PKL))
else:
    print(f'Pre-computed — {config.HITK_PARTITIONS_PKL.name} already exists, skipping save')

### Function
##### Takes in pos & neg train, learns model and makes forward pass

In [ ]:
raw_l3_recall = analyse_parameter_slices_quantile(inf_cv_gs_results_l3[0][1])[['min_param', 'max_param', 'recall_mean', 'recall_ci']]

In [ ]:
best_configs_l3 = find_best_configs_quantile(inf_cv_gs_results_l3, 'Gene_ADR_deg', 1)[['slice_num', 'param_combo', 'min_param', 'max_param', 'recall_mean', 'recall_ci']]
best_configs_l3

In [ ]:
from typing import List, Set, Tuple, Optional, Any

In [ ]:
def train_link_prediction_model(head_col: str, tail_col: str,
                                train_pos: pd.DataFrame,
                                train_neg: pd.DataFrame,
                                neg_to_test: pd.DataFrame,
                                test_pos_pairs: pd.DataFrame,
                                cols_to_drop: Optional[List[str]] = None,
                                **rf_kwargs: Any
) -> Tuple[RandomForestClassifier, pd.DataFrame]:
    """
    Trains a Random Forest model for link prediction and prepares the test set.

    Args:
        train_pos (pd.DataFrame): DataFrame of positive training examples.
        train_neg (pd.DataFrame): DataFrame of negative training examples.
        neg_to_test (pd.DataFrame): DataFrame of candidate pairs for evaluation.
        test_pos_pairs (pd.DataFrame): A DataFrame with 'head' and 'tail' columns
                                       representing the ground-truth positive pairs.
        cols_to_drop (Optional[List[str]], optional): Additional feature
            columns to drop before training. Defaults to None.
        **rf_kwargs: Optional keyword arguments to pass to the
                     RandomForestClassifier (e.g., max_depth, n_jobs).

    Returns:
        Tuple[RandomForestClassifier, pd.DataFrame]: A tuple containing:
            - The trained scikit-learn model object.
            - The neg_to_test DataFrame, now with 'true_label' and
              'model_score' columns, ready for evaluation.
    """
    # --- 1. Prepare and Check the Test Data ---
    # Convert the ground-truth DataFrame to a set of tuples for efficient lookup
    ground_truth_set = set(zip(test_pos_pairs[head_col], test_pos_pairs[tail_col]))

    # Create a set of pairs from the test candidates
    test_candidate_pairs = set(zip(neg_to_test[head_col], neg_to_test[tail_col]))

    # Check if all ground-truth pairs are present in the candidate set
    if not ground_truth_set.issubset(test_candidate_pairs):
        missing_pairs = len(ground_truth_set - test_candidate_pairs)
        raise ValueError(
            f"{missing_pairs} pairs from 'test_pos_pairs' are missing "
            "from the 'neg_to_test' candidate DataFrame."
        )

    # --- 2. Prepare Training Data ---
    train_pos = train_pos.assign(label=1)
    train_neg = train_neg.assign(label=0)
    train_df = pd.concat([train_pos, train_neg], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    final_cols_to_drop = [head_col, tail_col] + (cols_to_drop or [])
    feature_cols = [col for col in train_df.columns if col not in final_cols_to_drop + ['label']]
    X_train = train_df[feature_cols]
    y_train = train_df['label']

    # --- 3. Train the Random Forest Model ---
    model = RandomForestClassifier(n_estimators=200, random_state=69, **rf_kwargs)
    print("Training Random Forest model...")
    model.fit(X_train, y_train)
    print("Training complete.")

    # --- 4. Prepare Test Set and Predict Scores ---
    eval_df = neg_to_test.copy()
    eval_df['true_label'] = eval_df.apply(
        lambda row: (row[head_col], row[tail_col]) in ground_truth_set,
        axis=1
    )
    X_test = eval_df[feature_cols]
    prob_scores = model.predict_proba(X_test)[:, 1]
    eval_df['model_score'] = prob_scores

    return model, eval_df[[head_col, tail_col, 'true_label', 'model_score']]

### INF

In [ ]:
test_pos_pairs_load = load_pickle(str(config.TEST_POS_PAIRS_PKL))

In [ ]:
from pathlib import Path as _Path
if _Path(config.HITK_DFS_PKL).exists():
    hitk_dfs = load_pickle(str(config.HITK_DFS_PKL))
    print(f'Pre-computed — loaded {config.HITK_DFS_PKL.name}')
else:
    hitk_dfs = {combo_idx: None for combo_idx, _ in enumerate(toolbox.param_combos)}

    for combo_idx, combo in enumerate(toolbox.param_combos):

        print(f'Now preparing combo {combo_idx}: ({combo})')

        hitk_dfs[combo_idx] = train_link_prediction_model('Gene', 'ADR',
                                                          inf_hitk_train_test_feats['train_pos'][combo_idx],
                                                          inf_hitk_train_test_feats['train_neg'][combo_idx],
                                                          inf_hitk_train_test_feats['negs_to_test'][combo_idx],
                                                          test_pos_pairs_load,
                                                          ['Gene_ADR_deg'],
                                                          **inf_cv_gs_results_l3[combo_idx][2])

### DWPC baseline model

In [ ]:
dwpc_negs_to_test_dir = str(config.DWPC_FEATS_DIR)
dwpc_negs_to_test = load_pickle(f'{dwpc_negs_to_test_dir}/negs_to_test_l3_feats_ids.pkl')

In [ ]:
dwpc_pos_train = load_pickle(f'{dwpc_negs_to_test_dir}/train_pos_dwpc.pkl')
dwpc_neg_train = load_pickle(f'{dwpc_negs_to_test_dir}/train_neg_dwpc.pkl')

In [ ]:
dwpc_pos_train.drop(columns=['Label', 'PairID'], inplace=True)

In [ ]:
dwpc_neg_train.drop(columns=['Label', 'PairID'], inplace=True)

In [ ]:
dwpc_negs_to_test.pop('Label')

In [ ]:
len(dwpc_pos_train.columns) == len(dwpc_neg_train.columns) == len(dwpc_negs_to_test.columns)

In [ ]:
dwpc_hitk_train_test_feats = {'train_pos': dwpc_pos_train,
                            'train_neg': dwpc_neg_train,
                            'negs_to_test': dwpc_negs_to_test}

In [ ]:
# Build gene degree lookup from the pre-computed INF partition (already has Gene_ADR_deg)
genes_degs_adr_df = inf_hitk_train_test_feats['negs_to_test'][0][['Gene', 'Gene_ADR_deg']].drop_duplicates('Gene')

for partition_key in dwpc_hitk_train_test_feats:
    if 'Gene_ADR_deg' in dwpc_hitk_train_test_feats[partition_key].columns:
        dwpc_hitk_train_test_feats[partition_key] = dwpc_hitk_train_test_feats[partition_key].drop(columns=['Gene_ADR_deg'])
    dwpc_hitk_train_test_feats[partition_key] = pd.merge(
        dwpc_hitk_train_test_feats[partition_key], genes_degs_adr_df, how='left', on='Gene')
    print(f'{partition_key}: Gene_ADR_deg merged in')

In [ ]:
dwpc_cv_gs_results_l3_recall = load_pickle(str(config.CV_GS_RESULTS_L3_DWPC_PKL))

In [ ]:
from pathlib import Path as _Path
if not _Path(config.HITK_DFS_PKL).exists():
    hitk_dfs['dwpc'] = train_link_prediction_model('Gene', 'ADR',
                                                          dwpc_hitk_train_test_feats['train_pos'],
                                                          dwpc_hitk_train_test_feats['train_neg'],
                                                          dwpc_hitk_train_test_feats['negs_to_test'],
                                                          test_pos_pairs_load,
                                                          ['Gene_ADR_deg'],
                                                          **dwpc_cv_gs_results_l3_recall[2])
else:
    print(f'Pre-computed — skipping ({config.HITK_DFS_PKL.name} exists)')

### Compute overall and degree-stratified evaluation metrics

In [ ]:
bounds_data = {
    'min_param': [0, 6],
    'max_param': [5, 17]
}
ae_deg_boundaries = pd.DataFrame(bounds_data)

In [ ]:
def calculate_hits_at_k(df: pd.DataFrame, k: int, head_col, tail_col, n_bootstrap: int = 1000) -> Tuple[float, Tuple[float, float]]:
    """
    Calculates Hits@k and its 95% bootstrap confidence interval.
    """
    all_filtered_ranks = []
    
    if df.empty:
        return 0.0, (0.0, 0.0)
        
    for head, group in df.groupby(head_col):
        sorted_group = group.sort_values('model_score', ascending=False)
        true_tails = set(sorted_group[sorted_group['true_label'] == True][tail_col])
        
        if not true_tails:
            continue
            
        ranked_tails = sorted_group[tail_col].tolist()
        for true_t in true_tails:
            try:
                raw_rank_idx = ranked_tails.index(true_t)
            except ValueError:
                continue
                
            num_other_true_tails_ranked_higher = len([
                t for t in ranked_tails[:raw_rank_idx] if t in true_tails
            ])
            filtered_rank = (raw_rank_idx + 1) - num_other_true_tails_ranked_higher
            all_filtered_ranks.append(filtered_rank)
            
    filtered_ranks = np.array(all_filtered_ranks)
    
    if filtered_ranks.size == 0:
        return 0.0, (0.0, 0.0)
        
    hits_k_score = np.mean(filtered_ranks <= k)
    
    bootstrap_samples = np.random.choice(filtered_ranks, (n_bootstrap, len(filtered_ranks)), replace=True)
    bootstrap_means = np.mean(bootstrap_samples <= k, axis=1)
    ci_lower = np.percentile(bootstrap_means, 2.5)
    ci_upper = np.percentile(bootstrap_means, 97.5)
    
    return hits_k_score, (ci_lower, ci_upper)

In [ ]:
def calculate_recall(df: pd.DataFrame, head_col: str, tail_col: str, n_bootstrap: int = 1000) -> Tuple[float, Tuple[float, float]]:
    """
    Calculates recall and its 95% bootstrap confidence interval.
    
    Recall is defined as: True Positives / (True Positives + False Negatives)
    A positive prediction is defined by a score >= 0.5.
    """
    all_positive_predictions = []
    
    if df.empty:
        return 0.0, (0.0, 0.0)
        
    for head, group in df.groupby(head_col):
        # Identify the true positive pairs for this head entity
        true_positives_in_group = group[group['true_label'] == True]
        
        # A positive prediction is where the model_score is >= 0.5
        predicted_positives_in_group = group[group['model_score'] >= 0.5]
        
        # True Positives (TP): A true label and a positive prediction
        true_positives = len(predicted_positives_in_group.merge(
            true_positives_in_group, 
            on=[head_col, tail_col], 
            how='inner'
        ))
        
        # False Negatives (FN): A true label but a negative prediction (score < 0.5)
        # These are the true positives that were not in the predicted positives
        false_negatives = len(true_positives_in_group) - true_positives
        
        # Store the individual results for this group
        if (true_positives + false_negatives) > 0:
            all_positive_predictions.append({
                'true_positives': true_positives,
                'false_negatives': false_negatives
            })

    if not all_positive_predictions:
        return 0.0, (0.0, 0.0)
    
    # Bootstrap for confidence interval
    recall_scores = []
    sample_indices = np.arange(len(all_positive_predictions))
    
    for _ in range(n_bootstrap):
        bootstrap_indices = np.random.choice(sample_indices, size=len(sample_indices), replace=True)
        bootstrap_sample = [all_positive_predictions[i] for i in bootstrap_indices]
        
        total_tp = sum(item['true_positives'] for item in bootstrap_sample)
        total_fn = sum(item['false_negatives'] for item in bootstrap_sample)
        
        if (total_tp + total_fn) > 0:
            bootstrap_recall = total_tp / (total_tp + total_fn)
            recall_scores.append(bootstrap_recall)
        else:
            recall_scores.append(0.0)

    # Calculate overall recall and confidence interval
    recall_score = np.mean(recall_scores)
    ci_lower = np.percentile(recall_scores, 2.5)
    ci_upper = np.percentile(recall_scores, 97.5)
    
    return recall_score, (ci_lower, ci_upper)

In [ ]:
def train_and_predict_single_model(
    train_pos: pd.DataFrame, 
    train_neg: pd.DataFrame, 
    eval_candidates: pd.DataFrame, 
    test_pos_pairs: pd.DataFrame, 
    head_col: str, 
    tail_col: str, 
    model_params: Dict[str, Any], 
    cols_to_drop: List[str]
) -> pd.DataFrame:
    """
    Trains a single Random Forest model with given parameters and returns
    predictions on the evaluation set.
    """
    # --- 1. Prepare and Check the Test Data ---
    ground_truth_set = set(zip(test_pos_pairs[head_col], test_pos_pairs[tail_col]))
    test_candidate_pairs = set(zip(eval_candidates[head_col], eval_candidates[tail_col]))

    if not ground_truth_set.issubset(test_candidate_pairs):
        missing_pairs = len(ground_truth_set - test_candidate_pairs)
        raise ValueError(
            f"{missing_pairs} pairs from 'test_pos_pairs' are missing "
            "from the 'eval_candidates' candidate DataFrame."
        )

    # --- 2. Prepare Training Data ---
    train_pos = train_pos.assign(label=1)
    train_neg = train_neg.assign(label=0)
    train_df = pd.concat([train_pos, train_neg], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    final_cols_to_drop = [head_col, tail_col] + (cols_to_drop or [])
    feature_cols = [col for col in train_df.columns if col not in final_cols_to_drop + ['label']]
    X_train = train_df[feature_cols]
    y_train = train_df['label']

    # --- 3. Train the Random Forest Model ---
    model = RandomForestClassifier(n_estimators=200, random_state=69, **model_params)
    model.fit(X_train, y_train)

    # --- 4. Prepare Test Set and Predict Scores ---
    eval_df = eval_candidates.copy()
    eval_df['true_label'] = eval_df.apply(
        lambda row: (row[head_col], row[tail_col]) in ground_truth_set,
        axis=1
    )
    X_test = eval_df[feature_cols]
    prob_scores = model.predict_proba(X_test)[:, 1]
    eval_df['model_score'] = prob_scores

    return eval_df

In [ ]:
def ensemble_evaluation_by_quantile(
    best_configs: pd.DataFrame, 
    train_test_feats: Dict[str, Dict[str, pd.DataFrame]],
    test_pos_pairs: pd.DataFrame,
    grid_search_results: Dict[str, Any],
    head_col: str, 
    tail_col: str, 
    param_col, cols_to_drop: List[str], 
    hits_k_vals: List[int],
    n_bootstrap: int = 1000
) -> Dict[str, Any]:
    """
    Trains and evaluates an ensemble of models, each optimized for a specific
    quantile of a feature, using a different feature matrix for each model config.
    """
    
    # 1. Clean up quantile boundaries
    best_configs = best_configs.sort_values('slice_num').reset_index(drop=True)

    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        if best_configs.loc[i, 'min_param'] != prev_max + 1:
            best_configs.loc[i, 'min_param'] = prev_max + 1
    best_configs.loc[len(best_configs) - 1, 'max_param'] = np.inf

    # 2. Iterate through each quantile, train, and predict
    all_predictions = pd.DataFrame()
    results = {}

    test_pos_pairs_with_param = pd.merge(
    test_pos_pairs, train_test_feats['negs_to_test'][best_configs['param_combo'].iloc[0]][[head_col, tail_col, param_col]].drop_duplicates(), 
    on=[head_col, tail_col], 
    how='left'
)
    for idx, row in tqdm(best_configs.iterrows(), total=len(best_configs)):
        slice_num = row['slice_num']
        min_param, max_param = row['min_param'], row['max_param']

        config_key = row['param_combo']
        
        try:
            model_params = grid_search_results[config_key][2]
            # Fetch the FULL training and test data for this config
            train_pos = train_test_feats['train_pos'][config_key]
            train_neg = train_test_feats['train_neg'][config_key]
            eval_candidates = train_test_feats['negs_to_test'][config_key]
        except KeyError:
            print(f"Warning: Data or hyperparameters not found for config '{config_key}'. Skipping.")
            continue

        # Filter test candidates and true positive pairs for the current quantile

        if int(slice_num) == 10:
            test_slice_condition = (eval_candidates[param_col] >= min_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param)
        else:
            test_slice_condition = (eval_candidates[param_col] >= min_param) & (eval_candidates[param_col] <= max_param)
            test_pos_slice_condition = (test_pos_pairs_with_param[param_col] >= min_param) & (test_pos_pairs_with_param[param_col] <= max_param)
        
        test_slice = eval_candidates[test_slice_condition].copy()
        test_pos_slice = test_pos_pairs_with_param[test_pos_slice_condition].copy()

        # Train and predict
        if not test_slice.empty:
            slice_predictions = train_and_predict_single_model(
                train_pos=train_pos,         # Pass the FULL training data
                train_neg=train_neg,         # Pass the FULL training data
                eval_candidates=test_slice,  # Evaluate on the SLICED test data
                test_pos_pairs=test_pos_slice, # Evaluate on the SLICED true positives
                head_col=head_col, 
                tail_col=tail_col, 
                model_params=model_params, 
                cols_to_drop=cols_to_drop
            )
            all_predictions = pd.concat([all_predictions, slice_predictions], ignore_index=True)
            
            # Evaluate individual slice
            for k in hits_k_vals:
                hits_k, hits_k_ci = calculate_hits_at_k(slice_predictions, k, head_col, tail_col)
                results[f'slice_{slice_num}_Hits@{k}'] = hits_k
                results[f'slice_{slice_num}_Hits@{k}_ci'] = hits_k_ci
            
            # NEW: Calculate recall for the individual slice
            # The function now uses the 'true_label' column already in slice_predictions
            slice_recall, slice_recall_ci = calculate_recall(slice_predictions, head_col, tail_col)
            results[f'slice_{slice_num}_Recall'] = slice_recall
            results[f'slice_{slice_num}_Recall_ci'] = slice_recall_ci
    
    # 3. Evaluate Ensemble
    for k in hits_k_vals:
        hits_k, hits_k_ci = calculate_hits_at_k(all_predictions, k, head_col, tail_col)
        results[f'ensemble_Hits@{k}'] = hits_k
        results[f'ensemble_Hits@{k}_ci'] = hits_k_ci
    
    # NEW: Calculate recall for the entire ensemble
    # The function now uses the 'true_label' column already in all_predictions
    ensemble_recall, ensemble_recall_ci = calculate_recall(all_predictions, head_col, tail_col)
    results['ensemble_Recall'] = ensemble_recall
    results['ensemble_Recall_ci'] = ensemble_recall_ci
        
    return results, all_predictions

### INF ensemble & INF single best

In [ ]:
test_pos_pairs_load = load_pickle(str(config.TEST_POS_PAIRS_PKL))

In [ ]:
inf_hitk_train_test_feats = load_pickle(str(config.HITK_PARTITIONS_PKL))

In [ ]:
hitk_inf_ensemble, preds_inf_ensemble = ensemble_evaluation_by_quantile(best_configs_l3, inf_hitk_train_test_feats, test_pos_pairs_load, inf_cv_gs_results_l3, 'Gene', 'ADR', 'Gene_ADR_deg', ['Gene_ADR_deg'], [10,20])

In [ ]:
from pathlib import Path as _Path
if _Path(config.RAW_PREDS_PKL).exists():
    raw_preds = load_pickle(str(config.RAW_PREDS_PKL))
    print(f'Pre-computed — loaded {config.RAW_PREDS_PKL.name}')
else:
    raw_preds = train_and_predict_single_model(inf_hitk_train_test_feats['train_pos'][0], inf_hitk_train_test_feats['train_neg'][0],
                                                inf_hitk_train_test_feats['negs_to_test'][0], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                inf_cv_gs_results_l3[0][2], ['Gene_ADR_deg'])

In [ ]:
if not (config.RAW_PREDS_PKL).exists():
    save_pickle(raw_preds, str(config.RAW_PREDS_PKL))
else:
    print(f'Pre-computed — {config.RAW_PREDS_PKL.name} already exists, skipping save')

In [ ]:
from pathlib import Path as _Path
if _Path(config.DWPC_PREDS_PKL).exists():
    dwpc_preds = load_pickle(str(config.DWPC_PREDS_PKL))
    print(f'Pre-computed — loaded {config.DWPC_PREDS_PKL.name}')
else:
    dwpc_preds = train_and_predict_single_model(dwpc_hitk_train_test_feats['train_pos'], dwpc_hitk_train_test_feats['train_neg'],
                                                dwpc_hitk_train_test_feats['negs_to_test'], 
                                                test_pos_pairs_load, 
                                                'Gene', 'ADR', 
                                                dwpc_cv_gs_results_l3_recall[2], ['Gene_ADR_deg'])

In [ ]:
if not (config.DWPC_PREDS_PKL).exists():
    save_pickle(dwpc_preds, str(config.DWPC_PREDS_PKL))
else:
    print(f'Pre-computed — {config.DWPC_PREDS_PKL.name} already exists, skipping save')

In [ ]:
feats_l3 = load_pickle(str(config.DRKG_FEATS_L3_PKL))

In [ ]:
all_negs_to_test = {combo_idx: neg_to_test for combo_idx, neg_to_test in inf_hitk_train_test_feats['negs_to_test'].items()}
all_negs_to_test['dwpc'] = dwpc_hitk_train_test_feats['negs_to_test']

In [ ]:
from typing import Callable, Dict, List
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

In [ ]:
def permimp_ranking(model, X, y_true, feat_names, scorer):
    """
    Calculates permutation importance and returns the top 10 ranked features.
    
    Args:
        model: A trained model.
        X: DataFrame of features.
        y_true: Series of true labels.
        feat_names: List of feature names.
        scorer: The metric to use for scoring.
    
    Returns:
        A dictionary of the top 10 features and their importance scores.
    """
    
    permimp = permutation_importance(model, X, \
                       y_true, scoring=scorer, 
                       n_repeats=5, random_state=69, n_jobs=-1)['importances_mean']
        
    permimp_dict = {}
    for i in range(len(permimp)):
        permimp_dict[i] = permimp[i]
    
    permimp_rankings = {}
    
    for feat_index in sorted(permimp_dict, 
                             key=lambda dict_key: permimp_dict[dict_key], 
                             reverse=True)[:10]:
        permimp_rankings[feat_names[feat_index]] = round(permimp_dict[feat_index], 3)
        
    return permimp_rankings

In [ ]:
def get_sliced_permutation_importance_hitk(
    model: Callable,
    eval_df: pd.DataFrame,
    neg_to_test: pd.DataFrame,
    boundaries_df: pd.DataFrame,
    feature_cols: List[str],
    param_col: str,
    head_col: str,
    tail_col: str,
    k: int = 10,
    optimization_metric: str = 'hits@k',
    top_n_features: int = 5
) -> Dict[int, pd.DataFrame]:
    """
    Calculates permutation importance for features on different data slices, ensuring
    all data conforms to slice boundaries and the param_col is not dropped.
    """
    
    # Check for param_col existence to prevent KeyError
    if param_col not in eval_df.columns:
        raise KeyError(f"'{param_col}' not found in 'eval_df'. It is required for slicing.")
    
    if param_col not in neg_to_test.columns:
        raise KeyError(f"'{param_col}' not found in 'neg_to_test'. It is required for merging/slicing.")

    # Apply the boundary adjustment logic as requested, ensuring contiguous ranges
    best_configs = boundaries_df.copy()
    for i in range(1, len(best_configs)):
        prev_max = best_configs.loc[i - 1, 'max_param']
        best_configs.loc[i, 'min_param'] = prev_max + 1

    # Merge dataframes once to get a complete view
    full_eval_data = pd.merge(eval_df, neg_to_test, on=[head_col, tail_col, param_col], how='inner')
    
    if full_eval_data.empty:
        print("Warning: Merged evaluation data is empty. Check input dataframes and column names.")
        return {}

    results = {}
    
    # Define the scorer based on the chosen metric
    # The 'recall' scorer is correct as it is point-wise, so we keep the original logic.
    if optimization_metric == 'recall':
        scorer = 'recall'
    elif optimization_metric == 'hits@k':
        # For 'hits@k', we cannot use make_scorer. The logic must be handled manually
        # to ensure the necessary data (head, tail, etc.) is available for ranking.
        pass # The scorer is not needed here; logic is implemented below
    else:
        raise ValueError("optimization_metric must be 'hits@k' or 'recall'")

    # Iterate through each slice, slice the full dataframe, and run permutation importance
    for slice_idx, row in best_configs.iterrows():
        min_b, max_b = row['min_param'], row['max_param']

        # Slice the entire merged dataframe to ensure all data conforms to the boundaries
        slice_mask = (full_eval_data[param_col] >= min_b) & (full_eval_data[param_col] <= max_b)
        data_slice = full_eval_data.loc[slice_mask].copy()

        if data_slice.empty:
            continue

        # Isolate features and labels for the current slice
        X_slice = data_slice[feature_cols]
        y_slice = data_slice['true_label']

        # Use the correct logic for each metric
        if optimization_metric == 'recall':
            # This logic works for recall as it's a simple, point-wise metric
            importance_ranks = permimp_ranking(
                model=model,
                X=X_slice,
                y_true=y_slice,
                feat_names=feature_cols,
                scorer=scorer
            )
        elif optimization_metric == 'hits@k':
            # This manual logic correctly passes the necessary context for the hits@k scorer
            # as seen in the working code.
            base_scores = model.predict_proba(X_slice)[:, 1]
            scored_slice = data_slice.assign(model_score=base_scores)
            
            baseline_score = calculate_hits_at_k(scored_slice, k, head_col, tail_col)[0]
            
            importances = []
            for feature in feature_cols:
                X_shuffled = X_slice.copy()
                X_shuffled[feature] = np.random.permutation(X_shuffled[feature].values)
                
                shuffled_probs = model.predict_proba(X_shuffled)[:, 1]
                shuffled_scored_slice = data_slice.assign(model_score=shuffled_probs)
                
                score = calculate_hits_at_k(shuffled_scored_slice, k, head_col, tail_col)[0]
                
                importance = baseline_score - score
                importances.append({'feature': feature, 'importance': importance})
                
            importance_ranks = {item['feature']: item['importance'] for item in importances}

        # Format and store the results
        importance_df = pd.DataFrame(
            importance_ranks.items(), columns=['feature', 'importance']
        ).sort_values('importance', ascending=False).head(top_n_features).reset_index(drop=True)

        results[slice_idx] = importance_df
        
    return results

In [ ]:
def plot_hitk_ranked_bar_chart(results_df: pd.DataFrame, title: str, x_max_factor):
    """
    Generates a single horizontal bar chart showing a ranked list of models
    from a DataFrame of results, including bootstrap confidence intervals.

    Args:
        results_df (pd.DataFrame): A DataFrame with 'Model', 'Hits@10', 
                                   'Hits@10 CI lower', and 'Hits@10 CI upper'
                                   columns.
        title (str): The title for the chart.
    """
    # Create an 'error' column representing the margin of error
    # This is the difference between the mean and the lower CI bound
    results_df = results_df.copy()
    results_df['error'] = results_df['Hits@10'] - results_df['Hits@10 CI lower']
    
    # Sort the DataFrame by the main metric
    sorted_df = results_df.sort_values('Hits@10', ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    # The 'xerr' argument now uses the calculated error margin
    bars = ax.barh(sorted_df['model'], sorted_df['Hits@10'], 
                   xerr=sorted_df['error'], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=18)
    ax.set_xlabel('Hits@10', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=12)
    
    # Adjust x-axis limits to prevent cutoff and show the full error bar
    # Use the max of both the value and the upper CI for a correct limit
    x_max = sorted_df['Hits@10 CI upper'].max()
    ax.set_xlim(left=0, right=x_max * x_max_factor)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_recall_ranked_bar_chart(results_df: pd.DataFrame, title: str, x_max_factor): # recall
    """
    Generates a single horizontal bar chart showing a ranked list of models
    from a DataFrame of results, including bootstrap confidence intervals.
    """
    # Create an 'error' column representing the margin of error
    # This is the difference between the mean and the lower CI bound
    results_df = results_df.copy()
    results_df['error'] = results_df['Recall'] - results_df['Recall CI lower']
    
    # Sort the DataFrame by the main metric
    sorted_df = results_df.sort_values('Recall', ascending=True)
    
    # 3. Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.get_cmap('winter', len(sorted_df))

    # Create the horizontal bar plot with error bars
    # The 'xerr' argument now uses the calculated error margin
    bars = ax.barh(sorted_df['model'], sorted_df['Recall'], 
                   xerr=sorted_df['error'], capsize=5,
                   color=colors(np.linspace(0.25, 0.9, len(sorted_df))))
    
    # 4. Format the chart
    ax.set_title(title, fontsize=18)
    ax.set_xlabel('TPR', fontsize=14)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=12)
    
    # Adjust x-axis limits to prevent cutoff and show the full error bar
    # Use the max of both the value and the upper CI for a correct limit
    x_max = sorted_df['Recall CI upper'].max()
    ax.set_xlim(left=0, right=x_max * x_max_factor)

    plt.tight_layout()
    plt.show()

# III. Soldatos

## Preparation

In [ ]:
soldatos2_df = pd.read_csv(str(config.SOLDATOS_FILE_2), sep='\t')

In [ ]:
soldatos3_df = pd.read_csv(str(config.SOLDATOS_FILE_3), sep='\t')

In [ ]:
soldatos4_df = pd.read_csv(str(config.SOLDATOS_FILE_4), sep='\t')

In [ ]:
soldatos_df = pd.concat([
    soldatos2_df.loc[soldatos2_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05],
    soldatos3_df.loc[soldatos3_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05],
    soldatos4_df.loc[soldatos4_df['Q-Value (Based on Fisher Exact Test)'] <= 0.05]

])

soldatos_df.shape

In [ ]:
soldatos_gold = soldatos_df.loc[(soldatos_df['NUM_CASES_TARGET_AND_REACTION'] >= 10) & (soldatos_df['PRR'] >= 2)]

In [ ]:
map_ = pd.read_csv(str(config.MAPPING_FILE), sep='\t')

In [ ]:
hitk_dfs_mapped = {cfg_key: None for cfg_key in hitk_dfs}

for cfg_key in tqdm(hitk_dfs, total=len(hitk_dfs)):

    hitk_dfs_mapped[cfg_key] = hitk_dfs[cfg_key][1].merge(map_, left_on='Gene', right_on='id').drop(['id', 'kind'], axis=1).rename(columns={'name':'Gene_name'})

    hitk_dfs_mapped[cfg_key] = hitk_dfs_mapped[cfg_key].merge(map_, left_on='ADR', right_on='id', how='left').drop_duplicates().drop(['id','kind'], axis=1).rename(columns={'name':'ADR_name'})

In [ ]:
soldatos_gold['ADR_name'] = soldatos_gold['REACTION_NAME']

soldatos_gold['ADR_name']

In [ ]:
def filter_and_process_series(df, col_name, newcol_name, suffix):
    """
    Filters a DataFrame to keep rows where a specific column's values
    end with a given suffix, and creates a new series with the suffix removed.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to check for the suffix.
        suffix (str): The suffix to check for and remove.

    Returns:
        tuple: A tuple containing:
            - A new DataFrame with only the rows that had the suffix.
            - A new Series with the suffix removed from the entries.
    """
    # Create a boolean mask to identify rows where the series entry ends with the suffix
    has_suffix_mask = df[col_name].str.endswith(suffix)
    
    # Filter the DataFrame using the mask to keep only the relevant rows
    filtered_df = df[has_suffix_mask].copy()
    
    # Create a new series from the filtered DataFrame's column,
    # with the suffix removed from each entry
    filtered_df[newcol_name] = filtered_df[col_name].str.removesuffix(suffix)
    
    return filtered_df

In [ ]:
soldatos_gold_genes = filter_and_process_series(soldatos_gold, 'SWISSPROT_ID', 'Gene_name', '_HUMAN')

In [ ]:
def capitalise_column(df, column_name):
    """
    Capitalizes the first letter of each entry in a specified column of a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to be capitalized.

    Returns:
        pd.DataFrame: A new DataFrame with the specified column's entries capitalized.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    capitalized_df = df.copy()
    
    # Use the .str.capitalize() method to capitalize the first letter of each string
    # in the specified column
    capitalized_df[column_name] = capitalized_df[column_name].str.capitalize()
    
    return capitalized_df

In [ ]:
for cfg_key, df in tqdm(hitk_dfs_mapped.items(), total=len(hitk_dfs_mapped)):

    hitk_dfs_mapped[cfg_key] = capitalise_column(df, 'ADR_name')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_INTERSECTS_PKL).exists():
    intersects = load_pickle(str(config.SOLDATOS_INTERSECTS_PKL))
    new_test = load_pickle(str(config.SOLDATOS_NEW_TEST_PKL))
    print(f'Pre-computed — loaded both vars from existing pickles')
else:
    intersects = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    new_test = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    for cfg_key in tqdm(hitk_dfs_mapped, total=len(hitk_dfs_mapped)):

        intersects[cfg_key] = pd.merge(hitk_dfs_mapped[cfg_key], soldatos_gold_genes, on=['Gene_name', 'ADR_name'], how='inner')

        new_test[cfg_key] = intersects[cfg_key].loc[intersects[cfg_key]['true_label']==False]

In [ ]:
if not (config.SOLDATOS_INTERSECTS_PKL).exists():
    save_pickle(intersects, str(config.SOLDATOS_INTERSECTS_PKL))
    save_pickle(new_test, str(config.SOLDATOS_NEW_TEST_PKL))
else:
    print(f'Pre-computed — pickles already exist, skipping saves')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_HITK_MAPPED_PKL).exists():
    hitk_dfs_mapped_newtest = load_pickle(str(config.SOLDATOS_HITK_MAPPED_PKL))
    print(f'Pre-computed — loaded {config.SOLDATOS_HITK_MAPPED_PKL.name}')
else:
    hitk_dfs_mapped_newtest = {cfg_key: None for cfg_key in hitk_dfs_mapped}

    for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):

        hitk_dfs_mapped_newtest[cfg_key] = hitk_dfs_mapped[cfg_key][hitk_dfs_mapped[0]['true_label'] != True].copy()

        matched_keys = pd.merge(hitk_dfs_mapped_newtest[cfg_key], new_test[0], on=['Gene_name','ADR_name'], how='inner')

        matching_mask = pd.MultiIndex.from_frame(hitk_dfs_mapped_newtest[cfg_key][['Gene_name', 'ADR_name']]).isin(
            pd.MultiIndex.from_frame(matched_keys[['Gene_name', 'ADR_name']])
        )
    
        hitk_dfs_mapped_newtest[cfg_key].loc[matching_mask, 'true_label'] = True

In [ ]:
if not (config.SOLDATOS_HITK_MAPPED_PKL).exists():
    save_pickle(hitk_dfs_mapped_newtest, str(config.SOLDATOS_HITK_MAPPED_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_HITK_MAPPED_PKL.name} already exists, skipping save')

## Evaluating on Soldatos

In [ ]:
hitk_dfs_mapped_newtest = load_pickle(str(config.SOLDATOS_HITK_MAPPED_PKL))

In [ ]:
map_ = pd.read_csv(str(config.MAPPING_FILE), sep='\t')

In [ ]:
preds_inf_ensemble_mapped = preds_inf_ensemble[['Gene','ADR','Gene_ADR_deg','true_label','model_score']] # find the ensemble defined in internal testing

preds_inf_ensemble_mapped = preds_inf_ensemble_mapped.merge(map_, left_on='Gene', right_on='id').drop(['id', 'kind'], axis=1).rename(columns={'name':'Gene_name'})

preds_inf_ensemble_mapped = preds_inf_ensemble_mapped.merge(map_, left_on='ADR', right_on='id', how='left').drop_duplicates().drop(['id','kind'], axis=1).rename(columns={'name':'ADR_name'})

preds_inf_ensemble_mapped = capitalise_column(preds_inf_ensemble_mapped, 'ADR_name')
preds_inf_ensemble_mapped

In [ ]:
hitk_dfs_mapped_newtest['inf_ensemble'] = preds_inf_ensemble_mapped[preds_inf_ensemble_mapped['true_label'] != True].copy()

matched_keys = pd.merge(hitk_dfs_mapped_newtest['inf_ensemble'], new_test[0][['Gene_name','ADR_name']], on=['Gene_name','ADR_name'], how='inner')

matching_mask = pd.MultiIndex.from_frame(hitk_dfs_mapped_newtest['inf_ensemble'][['Gene_name', 'ADR_name']]).isin(
    pd.MultiIndex.from_frame(matched_keys[['Gene_name', 'ADR_name']])
)

hitk_dfs_mapped_newtest['inf_ensemble'].loc[matching_mask, 'true_label'] = True

In [ ]:
if not (config.SOLDATOS_PREDS_MAPPED_PKL).exists():
    save_pickle(hitk_dfs_mapped_newtest, str(config.SOLDATOS_PREDS_MAPPED_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_PREDS_MAPPED_PKL.name} already exists, skipping save')

In [ ]:
from pathlib import Path as _Path
if _Path(config.SOLDATOS_RESULTS_PKL).exists():
    newtest_overall_at10_ci = load_pickle(str(config.SOLDATOS_RESULTS_PKL))
    print(f'Pre-computed — loaded {config.SOLDATOS_RESULTS_PKL.name}')
else:
    newtest_overall_at10_ci = {cfg_key: {'Hits@10': None, 'Recall': None} for cfg_key in hitk_dfs_mapped_newtest} # default k is 10

    for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):

        newtest_overall_at10_ci[cfg_key]['Hits@10'] = calculate_hits_at_k(hitk_dfs_mapped_newtest[cfg_key], 10, 'Gene', 'ADR')

        newtest_overall_at10_ci[cfg_key]['Recall'] = calculate_recall(hitk_dfs_mapped_newtest[cfg_key], 'Gene', 'ADR')

In [ ]:
if not (config.SOLDATOS_RESULTS_PKL).exists():
    save_pickle(newtest_overall_at10_ci, str(config.SOLDATOS_RESULTS_PKL))
else:
    print(f'Pre-computed — {config.SOLDATOS_RESULTS_PKL.name} already exists, skipping save')

In [ ]:
def compute_sliced_hits_at_k_and_recall( # assumes no open-ended last boundary ie excludes top decile
    head_col: str, 
    tail_col: str,
    boundaries_df: pd.DataFrame,
    eval_df: pd.DataFrame,
    param_col_name: str,
    k: int = 10
) -> pd.DataFrame:
    """
    Computes Hits@k and Recall with CIs for slices of data defined by parameter
    boundaries, using a contiguous boundary adjustment.

    Args:
        head_col (str): Name of the head column.
        tail_col (str): Name of the tail column.
        boundaries_df (pd.DataFrame): DataFrame with 'min_param' and
                                      'max_param' columns defining the slices.
        eval_df (pd.DataFrame): The evaluation data, containing the parameter
                                column to slice on and model scores.
        param_col_name (str): The name of the column in eval_df to slice by.
        k (int, optional): The 'k' value for Hits@k. Defaults to 10.

    Returns:
        pd.DataFrame: A DataFrame with Hits@k, Recall, and their CIs for each slice.
    """
    if boundaries_df.empty:
        return pd.DataFrame()
    
    for i in range(1, len(boundaries_df)):
        prev_max = boundaries_df.loc[i - 1, 'max_param']
        if boundaries_df.loc[i, 'min_param'] != prev_max + 1:
            boundaries_df.loc[i, 'min_param'] = prev_max + 1

    results_list = []
    
    # Iterate through slices and compute metrics
    for idx, row in boundaries_df.iterrows():
        min_param = row['min_param']
        max_param = row['max_param']
        
        # Slicing with adjusted boundaries
        data_slice = eval_df[(eval_df[param_col_name] >= min_param) & (eval_df[param_col_name] <= max_param)]

        slice_result = {
            'slice_idx': idx,
            'min_param': min_param,
            'max_param': max_param
        }

        if not data_slice.empty:
            # Compute Hits@k and its CI
            hits_k, hits_k_ci = calculate_hits_at_k(data_slice, k, head_col, tail_col)
            slice_result[f'Hits@{k}'] = hits_k
            slice_result[f'Hits@{k}_ci_lower'] = hits_k_ci[0]
            slice_result[f'Hits@{k}_ci_upper'] = hits_k_ci[1]

            # Compute Recall and its CI
            recall, recall_ci = calculate_recall(data_slice, head_col, tail_col)
            slice_result['Recall'] = recall
            slice_result['Recall_ci_lower'] = recall_ci[0]
            slice_result['Recall_ci_upper'] = recall_ci[1]
        else:
            slice_result[f'Hits@{k}'] = np.nan
            slice_result[f'Hits@{k}_ci_lower'] = np.nan
            slice_result[f'Hits@{k}_ci_upper'] = np.nan
            slice_result['Recall'] = np.nan
            slice_result['Recall_ci_lower'] = np.nan
            slice_result['Recall_ci_upper'] = np.nan

        results_list.append(slice_result)

    return pd.DataFrame(results_list)

In [ ]:
bounds_data = {
    'min_param': [0, 6],
    'max_param': [5, 17]
}
ae_deg_boundaries = pd.DataFrame(bounds_data)

ae_deg_boundaries

In [ ]:
per_slice_at10_newtest = {cfg_key: None for cfg_key in hitk_dfs_mapped_newtest} # for k = 10

for cfg_key in tqdm(hitk_dfs_mapped_newtest, total=len(hitk_dfs_mapped_newtest)):
    
    per_slice_at10_newtest[cfg_key] = compute_sliced_hits_at_k_and_recall('Gene', 'ADR',
                                                           ae_deg_boundaries,
                                                           hitk_dfs_mapped_newtest[cfg_key],
                                                           'Gene_ADR_deg')

## Charts to plot from External testing

### Results to plot

In [ ]:
hitk_plot_data_external_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Hits@10': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][0],
                                            newtest_overall_at10_ci[0]['Hits@10'][0],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][0]],
                                'Hits@10 CI lower': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1][0],
                                            newtest_overall_at10_ci[0]['Hits@10'][1][0],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][1][0]],
                                'Hits@10 CI upper': [newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1][1],
                                            newtest_overall_at10_ci[0]['Hits@10'][1][1],
                                            newtest_overall_at10_ci['dwpc']['Hits@10'][1][1]]
                                }

pd.DataFrame(hitk_plot_data_external_test)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test), 'Hits @ 10 on Entire External Test Set', 3)

In [ ]:
recall_plot_data_external_test = {'model': ['INF_CV_Best\n(Ensemble)', 'Original', 'DWPC'],
                                'Recall': [newtest_overall_at10_ci['inf_ensemble']['Recall'][0],
                                            newtest_overall_at10_ci[0]['Recall'][0],
                                            newtest_overall_at10_ci['dwpc']['Recall'][0]],
                                'Recall CI lower': [newtest_overall_at10_ci['inf_ensemble']['Recall'][1][0],
                                            newtest_overall_at10_ci[0]['Recall'][1][0],
                                            newtest_overall_at10_ci['dwpc']['Recall'][1][0]],
                                'Recall CI upper': [newtest_overall_at10_ci['inf_ensemble']['Recall'][1][1],
                                            newtest_overall_at10_ci[0]['Recall'][1][1],
                                            newtest_overall_at10_ci['dwpc']['Recall'][1][1]]
                                }

pd.DataFrame(recall_plot_data_external_test)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test), 'TPR on Entire External Test Set', 1.75)

### Figure 11 — Permutation importance on Soldatos external test set

In [ ]:
perm_rankings_soldatos = {cfg_key: {'hits@10': None, 'recall': None} for cfg_key in [12, 0, 'dwpc']}

for cfg_key in tqdm([12, 0, 'dwpc'], total=3):
    
    print('doing hits@10...')
    perm_rankings_soldatos[cfg_key]['hits@10'] = get_sliced_permutation_importance_hitk(hitk_dfs[cfg_key][0], 
                                                                    hitk_dfs_mapped_newtest[cfg_key], 
                                                                    all_negs_to_test[cfg_key], 
                                                                    ae_deg_boundaries, 
                                                                    feats_l3,
                                                                    'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                    optimization_metric='hits@k')
    
    print('doing recall...')
    perm_rankings_soldatos[cfg_key]['recall'] = get_sliced_permutation_importance_hitk(hitk_dfs[cfg_key][0], 
                                                                hitk_dfs_mapped_newtest[cfg_key], 
                                                                all_negs_to_test[cfg_key], 
                                                                ae_deg_boundaries, 
                                                                feats_l3,
                                                                'Gene_ADR_deg', 'Gene', 'ADR', 
                                                                optimization_metric='recall')
    


In [ ]:
# Figure 11: Top-5 features by permutation importance (TPR, Soldatos external test, first decile 0-5 AEs)
pd.set_option('display.max_colwidth', None)
fig11_table = perm_rankings_soldatos[12]['recall'][0].copy()
fig11_table.index = range(1, 6)
fig11_table.columns = ['Metapath Feature', 'Importance (mean decrease in TPR)']
print("Figure 11: Top-5 features by permutation importance")
print("Config: INF-12 | Metric: TPR | Data: Soldatos external test | Decile: 1st (0-5 AEs per target)")
display(fig11_table)
pd.reset_option('display.max_colwidth')

In [ ]:
hitk_plot_data_external_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10'].values[0]],


                                'Hits@10 CI lower': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10_ci_lower'].values[0]],

                                            
                                'Hits@10 CI upper': [per_slice_at10_newtest[12].iloc[[0]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Hits@10_ci_upper'].values[0]],

                                }

pd.DataFrame(hitk_plot_data_external_test_d1)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test_d1), 'Hits @ 10 on External Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 3)

In [ ]:
recall_plot_data_external_test_d1 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [per_slice_at10_newtest[12].iloc[[0]]['Recall'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall'].values[0]],


                                'Recall CI lower': [per_slice_at10_newtest[12].iloc[[0]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall_ci_lower'].values[0]],

                                            
                                'Recall CI upper': [per_slice_at10_newtest[12].iloc[[0]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[0]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[0]]['Recall_ci_upper'].values[0]],

                                }

pd.DataFrame(recall_plot_data_external_test_d1)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test_d1), 'TPR on External Test Data with 0-5 AEs per Target\n(1st Decile Defined in Cross-Validation)', 2.5)

In [ ]:
hitk_plot_data_external_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Hits@10': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10'].values[0]],

                                'Hits@10 CI lower': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10_ci_lower'].values[0]],
                                            
                                'Hits@10 CI upper': [per_slice_at10_newtest[7].iloc[[1]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Hits@10_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Hits@10_ci_upper'].values[0]]
                                }

pd.DataFrame(hitk_plot_data_external_test_d2)

In [ ]:
plot_hitk_ranked_bar_chart(pd.DataFrame(hitk_plot_data_external_test_d2), 'Hits @ 10 on External Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)', 3)

In [ ]:
recall_plot_data_external_test_d2 = {'model': ['INF_CV_Best', 'Original', 'DWPC'],
                                'Recall': [per_slice_at10_newtest[7].iloc[[1]]['Recall'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall'].values[0]],

                                'Recall CI lower': [per_slice_at10_newtest[7].iloc[[1]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall_ci_lower'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall_ci_lower'].values[0]],
                                            
                                'Recall CI upper': [per_slice_at10_newtest[7].iloc[[1]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest[0].iloc[[1]]['Recall_ci_upper'].values[0],
                                            per_slice_at10_newtest['dwpc'].iloc[[1]]['Recall_ci_upper'].values[0]]
                                }

pd.DataFrame(recall_plot_data_external_test_d2)

In [ ]:
plot_recall_ranked_bar_chart(pd.DataFrame(recall_plot_data_external_test_d2), 'TPR on External Test Data with 6-17 AEs per Target\n(2nd Decile Defined in Cross-Validation)', 1.5)

## Figure S4 — Synthesis: Holdout vs. External Test Performance
Bar chart comparing Hits@10 and TPR for all three methods on the holdout and external (Soldatos) test sets.
Illustrates the collapse of hits@10 on external data contrasted with INF's resilient TPR.

In [ ]:
# ---------------------------------------------------------------------------
# Figure S4: Holdout vs. External Test Performance (synthesis plot)
# Compares Hits@10 and TPR for INF Ensemble, DWPC, and Original features
# across the holdout test set (n=998) and Soldatos external test set (n=7,804).
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Compute point estimates and CIs ---
raw_hitk_res  = calculate_hits_at_k(raw_preds,  10, 'Gene', 'ADR')
dwpc_hitk_res = calculate_hits_at_k(dwpc_preds, 10, 'Gene', 'ADR')
raw_rec_res   = calculate_recall(raw_preds,  'Gene', 'ADR')
dwpc_rec_res  = calculate_recall(dwpc_preds, 'Gene', 'ADR')

data = {
    'Hits@10': {
        'holdout': [
            (hitk_inf_ensemble['ensemble_Hits@10'],        hitk_inf_ensemble['ensemble_Hits@10_ci']),
            (dwpc_hitk_res[0],                               dwpc_hitk_res[1]),
            (raw_hitk_res[0],                                raw_hitk_res[1]),
        ],
        'external': [
            (newtest_overall_at10_ci['inf_ensemble']['Hits@10'][0], newtest_overall_at10_ci['inf_ensemble']['Hits@10'][1]),
            (newtest_overall_at10_ci['dwpc']['Hits@10'][0],         newtest_overall_at10_ci['dwpc']['Hits@10'][1]),
            (newtest_overall_at10_ci[0]['Hits@10'][0],               newtest_overall_at10_ci[0]['Hits@10'][1]),
        ],
    },
    'TPR': {
        'holdout': [
            (hitk_inf_ensemble['ensemble_Recall'],          hitk_inf_ensemble['ensemble_Recall_ci']),
            (dwpc_rec_res[0],                                 dwpc_rec_res[1]),
            (raw_rec_res[0],                                  raw_rec_res[1]),
        ],
        'external': [
            (newtest_overall_at10_ci['inf_ensemble']['Recall'][0], newtest_overall_at10_ci['inf_ensemble']['Recall'][1]),
            (newtest_overall_at10_ci['dwpc']['Recall'][0],         newtest_overall_at10_ci['dwpc']['Recall'][1]),
            (newtest_overall_at10_ci[0]['Recall'][0],               newtest_overall_at10_ci[0]['Recall'][1]),
        ],
    },
}

methods = ['INF Ensemble', 'DWPC', 'Original']
colors  = ['#1976D2', '#F57C00', '#757575']   # blue, orange, grey
x       = np.arange(len(methods))
width   = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=False)

for ax, metric in zip(axes, ['Hits@10', 'TPR']):
    hold_vals = [v for v, _ in data[metric]['holdout']]
    hold_errs = ([v - ci[0] for v, ci in data[metric]['holdout']],
                 [ci[1] - v for v, ci in data[metric]['holdout']])
    ext_vals  = [v for v, _ in data[metric]['external']]
    ext_errs  = ([v - ci[0] for v, ci in data[metric]['external']],
                 [ci[1] - v for v, ci in data[metric]['external']])

    bars1 = ax.bar(x - width / 2, hold_vals, width,
                   color=colors, alpha=0.90,
                   yerr=hold_errs, capsize=5,
                   ecolor='black', error_kw={'linewidth': 1.2, 'zorder': 5})
    bars2 = ax.bar(x + width / 2, ext_vals, width,
                   color=colors, alpha=0.45, hatch='//',
                   yerr=ext_errs, capsize=5,
                   ecolor='black', error_kw={'linewidth': 1.2, 'zorder': 5})

    # Annotate bar values
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8.5)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.12)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

# Shared legend (holdout = solid, external = hatched)
legend_elements = [
    mpatches.Patch(facecolor='grey', alpha=0.90,
                   label='Holdout test set (n = 998)'),
    mpatches.Patch(facecolor='grey', alpha=0.45, hatch='//',
                   label='External test set — Soldatos et al. (n = 7,804)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.06))

fig.suptitle(
    'Holdout vs. External Test Performance\n'
    'Hits@10 collapses on external data; INF TPR remains resilient',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(str(config.FIGS_DIR / 'figS4_holdout_vs_external.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure S4 saved to {config.FIGS_DIR / 'figS4_holdout_vs_external.png'}")

## Figures 7, 8, 9 — Per-decile performance (1st decile focus)

Reproduces the three bar-chart figures described in Results sections 3.2–3.4.

- **Figure 7**: CV TPR, 1st decile (0–5 AEs), Original vs DWPC vs Best INF
- **Figure 8**: Holdout test set, 1st decile — filtered Hits@10 and TPR
- **Figure 9**: External (Soldatos) test set — filtered Hits@10 and TPR

### Figure 7 — CV TPR, 1st decile

In [ ]:
# Figure 7 data — CV 1st decile (0-5 AEs per target) recall
# Best INF: slice_num==1 row of best_configs_l3 (already loaded)
inf5_mean = best_configs_l3[best_configs_l3['slice_num'] == 1].iloc[0]['recall_mean']
inf5_ci   = best_configs_l3[best_configs_l3['slice_num'] == 1].iloc[0]['recall_ci']

# Original: first row of raw_l3_recall (already computed)
orig5_mean = raw_l3_recall.iloc[0]['recall_mean']
orig5_ci   = raw_l3_recall.iloc[0]['recall_ci']

# DWPC: same analysis on the DWPC CV fold results
dwpc_l3_recall = analyse_parameter_slices_quantile(dwpc_cv_gs_results_l3_recall[1])[['min_param','max_param','recall_mean','recall_ci']]
dwpc5_mean = dwpc_l3_recall.iloc[0]['recall_mean']
dwpc5_ci   = dwpc_l3_recall.iloc[0]['recall_ci']

print(f'Original 1st decile CV TPR: {orig5_mean:.3f} ± {orig5_ci:.3f}')
print(f'DWPC     1st decile CV TPR: {dwpc5_mean:.3f} ± {dwpc5_ci:.3f}')
print(f'Best INF 1st decile CV TPR: {inf5_mean:.3f} ± {inf5_ci:.3f}')

In [ ]:
# ---------------------------------------------------------------------------
# Figure 7: CV TPR for 1st decile (0-5 AEs) — Original vs DWPC vs Best INF
# Two versions: vertical and horizontal bars.
# Accessibility palette: blue=INF, amber=DWPC, grey=Original
# ---------------------------------------------------------------------------
import matplotlib.patches as mpatches

COLORS5 = ['#999999', '#E69F00', '#0072B2']   # Original, DWPC, INF
methods5 = ['Original', 'DWPC', 'INF\n(Best Config)']
means5   = [orig5_mean, dwpc5_mean, inf5_mean]
cis5     = [orig5_ci,   dwpc5_ci,   inf5_ci]

def _style(ax, orient='v'):
    ax.spines[['top','right']].set_visible(False)
    if orient == 'v':
        ax.yaxis.grid(True, linestyle='--', alpha=0.45, zorder=0)
    else:
        ax.xaxis.grid(True, linestyle='--', alpha=0.45, zorder=0)
    ax.set_axisbelow(True)

# ── Vertical ──
fig, ax = plt.subplots(figsize=(6.5, 5))
x = np.arange(len(methods5))
bars = ax.bar(x, means5, width=0.55, color=COLORS5, alpha=0.88, zorder=3,
              yerr=[cis5, cis5], capsize=6,
              ecolor='#333333', error_kw={'linewidth':1.5,'zorder':4})
ax.set_ylim(0, max(means5)*1.35)
for bar, ci in zip(bars, cis5):
    ytext = bar.get_height() + ci + 0.02*ax.get_ylim()[1]
    ax.text(bar.get_x()+bar.get_width()/2, ytext,
            f'{bar.get_height():.2f}', ha='center', va='bottom',
            fontsize=9.5, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(methods5, fontsize=10.5)
ax.set_ylabel('Mean CV TPR', fontsize=11); ax.set_xlabel('Method', fontsize=11)
ax.set_title('Mean CV TPR — 1st Decile of Target–AE Associations\nby No. of Known AEs for Target (0–5 AEs), L3 Metapaths', fontsize=11.5, fontweight='bold', pad=10)
_style(ax, 'v')
plt.tight_layout()
plt.savefig(str(config.FIGS_DIR / 'fig7_cv_d1_tpr_vertical.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {config.FIGS_DIR / 'fig7_cv_d1_tpr_vertical.png'}")

# ── Horizontal ──
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(methods5))
bars = ax.barh(y, means5, height=0.55, color=COLORS5, alpha=0.88, zorder=3,
               xerr=[cis5, cis5], capsize=6,
               ecolor='#333333', error_kw={'linewidth':1.5,'zorder':4})
ax.set_xlim(0, max(means5)*1.35)
for bar, ci in zip(bars, cis5):
    xtext = bar.get_width() + ci + 0.025*ax.get_xlim()[1]
    ax.text(xtext, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.2f}', ha='left', va='center',
            fontsize=9.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(methods5, fontsize=10.5)
ax.set_xlabel('Mean CV TPR', fontsize=11); ax.set_ylabel('Method', fontsize=11)
ax.set_title('Mean CV TPR — 1st Decile of Target–AE Associations\nby No. of Known AEs for Target (0–5 AEs), L3 Metapaths', fontsize=11.5, fontweight='bold', pad=10)
_style(ax, 'h')
plt.tight_layout()
plt.savefig(str(config.FIGS_DIR / 'fig7_cv_d1_tpr_horizontal.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {config.FIGS_DIR / 'fig7_cv_d1_tpr_horizontal.png'}")


### Figure 8 — Holdout test set, 1st decile

In [ ]:
# Figure 8 data — holdout test set, 1st decile (0-5 AEs)
#
# INF (Best Config): TPR and CI from pre-computed ideal_inf_res_recall pickle
#   (config 12, deterministic stored result). Hits@10 from config-12 predictions
#   in inf_configs_slice_recall, filtered to d1.
# DWPC: computed from dwpc_preds filtered to d1. Note: ~0.255 (reproducible
#   from the current DWPC model); source notebook used an older 16-feature matrix
#   that gave 0.252 — the 0.003 difference is kept here for reproducibility.
# Original: TPR/CI from pre-computed inf_configs_slice_recall[0] (deterministic).

ideal_recall     = load_pickle(str(config.IDEAL_INF_RES_RECALL_PKL))
inf_slice_recall = load_pickle(str(config.INF_CONFIGS_SLICE_RECALL_PKL))

_c12_m, c12_preds = inf_slice_recall[12]
_c0_m,  _         = inf_slice_recall[0]

# INF — deterministic stored values
inf6_rec    = ideal_recall['best_slice_scores']['slice_1.0_Recall']['score']
inf6_rec_ci = ideal_recall['best_slice_scores']['slice_1.0_Recall']['ci']

# Original — deterministic stored values
orig6_rec    = _c0_m['slice_1.0_Recall']
orig6_rec_ci = _c0_m['slice_1.0_Recall_ci']

# Hits@10 from stored predictions (point estimate is deterministic)
inf6_d1  = c12_preds[c12_preds['Gene_ADR_deg'] <= 5].copy()
raw_d1   = raw_preds[raw_preds['Gene_ADR_deg']   <= 5].copy()
dwpc_d1  = dwpc_preds[dwpc_preds['Gene_ADR_deg'] <= 5].copy()

inf6_h10,  inf6_h10_ci  = calculate_hits_at_k(inf6_d1,  10, 'Gene', 'ADR')
orig6_h10, orig6_h10_ci = calculate_hits_at_k(raw_d1,   10, 'Gene', 'ADR')
dwpc6_h10, dwpc6_h10_ci = calculate_hits_at_k(dwpc_d1,  10, 'Gene', 'ADR')
dwpc6_rec, dwpc6_rec_ci = calculate_recall(   dwpc_d1,      'Gene', 'ADR')

print('1st decile holdout Hits@10:')
print(f'  Best INF Config: {inf6_h10:.3f} ({inf6_h10_ci[0]:.3f}, {inf6_h10_ci[1]:.3f})')
print(f'  DWPC:            {dwpc6_h10:.3f} ({dwpc6_h10_ci[0]:.3f}, {dwpc6_h10_ci[1]:.3f})')
print(f'  Original:        {orig6_h10:.3f} ({orig6_h10_ci[0]:.3f}, {orig6_h10_ci[1]:.3f})')
print('1st decile holdout TPR:')
print(f'  Best INF Config: {inf6_rec:.3f} ({inf6_rec_ci[0]:.3f}, {inf6_rec_ci[1]:.3f})')
print(f'  DWPC:            {dwpc6_rec:.3f} ({dwpc6_rec_ci[0]:.3f}, {dwpc6_rec_ci[1]:.3f})')
print(f'  Original:        {orig6_rec:.3f} ({orig6_rec_ci[0]:.3f}, {orig6_rec_ci[1]:.3f})')

In [ ]:
# ---------------------------------------------------------------------------
# Figure 8: Holdout test set, 1st decile — Hits@10 and TPR
# Two panels (left=Hits@10, right=TPR), two versions (vertical / horizontal).
# ---------------------------------------------------------------------------
METHOD_ORDER = ['Best INF Config', 'DWPC', 'Original']
COLORS6 = {'Best INF Config': '#0072B2', 'DWPC': '#E69F00', 'Original': '#999999'}

fig8_data = {
    'Hits10_mean': [inf6_h10,   dwpc6_h10,   orig6_h10],
    'Hits10_lo':   [inf6_h10  - inf6_h10_ci[0],  dwpc6_h10  - dwpc6_h10_ci[0],  orig6_h10  - orig6_h10_ci[0]],
    'Hits10_hi':   [inf6_h10_ci[1]  - inf6_h10,  dwpc6_h10_ci[1]  - dwpc6_h10,  orig6_h10_ci[1]  - orig6_h10],
    'TPR_mean':    [inf6_rec,   dwpc6_rec,   orig6_rec],
    'TPR_lo':      [inf6_rec  - inf6_rec_ci[0],   dwpc6_rec  - dwpc6_rec_ci[0],  orig6_rec  - orig6_rec_ci[0]],
    'TPR_hi':      [inf6_rec_ci[1]  - inf6_rec,   dwpc6_rec_ci[1]  - dwpc6_rec,  orig6_rec_ci[1]  - orig6_rec],
}
colors6 = [COLORS6[m] for m in METHOD_ORDER]

shared_lim6 = max(fig8_data['Hits10_mean'] + fig8_data['TPR_mean']) * 1.40
for orient in ('vertical', 'horizontal'):
    if orient == 'vertical':
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.5))
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

    for ax, (metric, means, lo, hi) in zip(axes, [
        ('Hits@10', fig8_data['Hits10_mean'], fig8_data['Hits10_lo'], fig8_data['Hits10_hi']),
        ('TPR',     fig8_data['TPR_mean'],    fig8_data['TPR_lo'],    fig8_data['TPR_hi']),
    ]):
        n = len(METHOD_ORDER)
        if orient == 'vertical':
            pos = np.arange(n)
            bars = ax.bar(pos, means, width=0.55, color=colors6, alpha=0.88, zorder=3,
                          yerr=[lo, hi], capsize=6, ecolor='#333333',
                          error_kw={'linewidth':1.5,'zorder':4})
            ax.set_ylim(0, shared_lim6)
            for bar, l, h in zip(bars, lo, hi):
                ytext = bar.get_height() + h + 0.025*ax.get_ylim()[1]
                ax.text(bar.get_x()+bar.get_width()/2, ytext,
                        f'{bar.get_height():.2f}', ha='center', va='bottom',
                        fontsize=9.5, fontweight='bold')
            ax.set_xticks(pos); ax.set_xticklabels(METHOD_ORDER, fontsize=10)
            ax.set_ylabel(metric, fontsize=11)
            _style(ax, 'v')
        else:
            pos = np.arange(n)
            bars = ax.barh(pos, means, height=0.55, color=colors6, alpha=0.88, zorder=3,
                           xerr=[lo, hi], capsize=6, ecolor='#333333',
                           error_kw={'linewidth':1.5,'zorder':4})
            ax.set_xlim(0, shared_lim6)
            for bar, h in zip(bars, hi):
                xtext = bar.get_width() + h + 0.03*ax.get_xlim()[1]
                ax.text(xtext, bar.get_y()+bar.get_height()/2,
                        f'{bar.get_width():.2f}', ha='left', va='center',
                        fontsize=9.5, fontweight='bold')
            ax.set_yticks(pos); ax.set_yticklabels(METHOD_ORDER, fontsize=10)
            ax.set_xlabel(metric, fontsize=11)
            _style(ax, 'h')
        ax.set_title(metric, fontsize=12, fontweight='bold')

    fig.suptitle(
        'Holdout Test Set — 1st Decile of Target\u2013AE Associations\n'
        'by No. of Known AEs for Target (0\u20135 AEs); Best INF Config doubles TPR, Hits@10 is similar',
        fontsize=11, fontweight='bold', y=1.03)
    legend_patches = [mpatches.Patch(color=COLORS6[m], alpha=0.88, label=m) for m in METHOD_ORDER]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5,-0.07))
    plt.tight_layout()
    fname = f'fig8_holdout_d1_{orient}.png'
    plt.savefig(str(config.FIGS_DIR / fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {config.FIGS_DIR / fname}")


### Figure 9 — External (Soldatos) test set

In [ ]:
# Figure 9 data — Soldatos overall hits@10 and TPR
# newtest_overall_at10_ci already populated by cell 84 above.
inf7_h10,  inf7_h10_ci  = newtest_overall_at10_ci['inf_ensemble']['Hits@10']
inf7_rec,  inf7_rec_ci  = newtest_overall_at10_ci['inf_ensemble']['Recall']
dwpc7_h10, dwpc7_h10_ci = newtest_overall_at10_ci['dwpc']['Hits@10']
dwpc7_rec, dwpc7_rec_ci = newtest_overall_at10_ci['dwpc']['Recall']
orig7_h10, orig7_h10_ci = newtest_overall_at10_ci[0]['Hits@10']
orig7_rec, orig7_rec_ci = newtest_overall_at10_ci[0]['Recall']

print('Soldatos overall Hits@10:')
print(f'  INF Ensemble: {inf7_h10:.4f} ({inf7_h10_ci[0]:.4f}, {inf7_h10_ci[1]:.4f})')
print(f'  DWPC:         {dwpc7_h10:.4f} ({dwpc7_h10_ci[0]:.4f}, {dwpc7_h10_ci[1]:.4f})')
print(f'  Original:     {orig7_h10:.4f} ({orig7_h10_ci[0]:.4f}, {orig7_h10_ci[1]:.4f})')
print('Soldatos overall TPR:')
print(f'  INF Ensemble: {inf7_rec:.3f} ({inf7_rec_ci[0]:.3f}, {inf7_rec_ci[1]:.3f})')
print(f'  DWPC:         {dwpc7_rec:.3f} ({dwpc7_rec_ci[0]:.3f}, {dwpc7_rec_ci[1]:.3f})')
print(f'  Original:     {orig7_rec:.3f} ({orig7_rec_ci[0]:.3f}, {orig7_rec_ci[1]:.3f})')

In [ ]:
# ---------------------------------------------------------------------------
# Figure 9: External validation (Soldatos et al., n=7,804) — Hits@10 and TPR
# Two panels (left=Hits@10, right=TPR), two versions (vertical / horizontal).
# ---------------------------------------------------------------------------
fig9_data = {
    'Hits10_mean': [inf7_h10,   dwpc7_h10,   orig7_h10],
    'Hits10_lo':   [inf7_h10  - inf7_h10_ci[0],  dwpc7_h10  - dwpc7_h10_ci[0],  orig7_h10  - orig7_h10_ci[0]],
    'Hits10_hi':   [inf7_h10_ci[1]  - inf7_h10,  dwpc7_h10_ci[1]  - dwpc7_h10,  orig7_h10_ci[1]  - orig7_h10],
    'TPR_mean':    [inf7_rec,   dwpc7_rec,   orig7_rec],
    'TPR_lo':      [inf7_rec  - inf7_rec_ci[0],   dwpc7_rec  - dwpc7_rec_ci[0],  orig7_rec  - orig7_rec_ci[0]],
    'TPR_hi':      [inf7_rec_ci[1]  - inf7_rec,   dwpc7_rec_ci[1]  - dwpc7_rec,  orig7_rec_ci[1]  - orig7_rec],
}

SHARED_LIM7 = 0.5  # shared axis limit for both panels — drives home the Hits@10 collapse

for orient in ('vertical', 'horizontal'):
    if orient == 'vertical':
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.5))
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

    for ax, (metric, means, lo, hi, fmt) in zip(axes, [
        ('Hits@10', fig9_data['Hits10_mean'], fig9_data['Hits10_lo'], fig9_data['Hits10_hi'], '{:.2f}'),
        ('TPR',     fig9_data['TPR_mean'],    fig9_data['TPR_lo'],    fig9_data['TPR_hi'],    '{:.2f}'),
    ]):
        n = len(METHOD_ORDER)
        if orient == 'vertical':
            pos = np.arange(n)
            bars = ax.bar(pos, means, width=0.55, color=colors6, alpha=0.88, zorder=3,
                          yerr=[lo, hi], capsize=6, ecolor='#333333',
                          error_kw={'linewidth':1.5,'zorder':4})
            ax.set_ylim(0, SHARED_LIM7)
            for bar, l, h in zip(bars, lo, hi):
                ytext = bar.get_height() + h + 0.025*ax.get_ylim()[1]
                ax.text(bar.get_x()+bar.get_width()/2, ytext,
                        fmt.format(bar.get_height()), ha='center', va='bottom',
                        fontsize=9.5, fontweight='bold')
            ax.set_xticks(pos); ax.set_xticklabels(METHOD_ORDER, fontsize=10)
            ax.set_ylabel(metric, fontsize=11)
            _style(ax, 'v')
        else:
            pos = np.arange(n)
            bars = ax.barh(pos, means, height=0.55, color=colors6, alpha=0.88, zorder=3,
                           xerr=[lo, hi], capsize=6, ecolor='#333333',
                           error_kw={'linewidth':1.5,'zorder':4})
            ax.set_xlim(0, SHARED_LIM7)
            for bar, h in zip(bars, hi):
                xtext = bar.get_width() + h + 0.03*ax.get_xlim()[1]
                ax.text(xtext, bar.get_y()+bar.get_height()/2,
                        fmt.format(bar.get_width()), ha='left', va='center',
                        fontsize=9.5, fontweight='bold')
            ax.set_yticks(pos); ax.set_yticklabels(METHOD_ORDER, fontsize=10)
            ax.set_xlabel(metric, fontsize=11)
            _style(ax, 'h')
        ax.set_title(metric, fontsize=12, fontweight='bold')

    fig.suptitle(
        'External Validation \u2014 Soldatos et al. Test Set (n = 7,804)\n'
        'Hits@10 collapses for all methods; INF TPR remains resilient',
        fontsize=11, fontweight='bold', y=1.03)
    legend_patches = [mpatches.Patch(color=COLORS6[m], alpha=0.88, label=m) for m in METHOD_ORDER]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5,-0.07))
    plt.tight_layout()
    fname = f'fig9_soldatos_{orient}.png'
    plt.savefig(str(config.FIGS_DIR / fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {config.FIGS_DIR / fname}")